# ЛР 4.2. Упрощённый детектор объектов (YOLO-like)

Синтетические фигуры → CNN → класс + bounding box → IoU и визуализация.


### Цель

Познакомиться с одностадийной детекцией: сеть предсказывает класс и рамку за один проход.


In [ ]:
# !pip install torch matplotlib opencv-python pillow

import math
import random
from dataclasses import dataclass

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Быстрый учебный режим: меньше картинка/эпохи, но с нормальным качеством
IMG_SIZE = 96
NUM_CLASSES = 3  # circle, rectangle, triangle
CLASS_NAMES = ['circle', 'rectangle', 'triangle']
BATCH_SIZE = 96
EPOCHS = 10
LR = 1e-3
LAMBDA_CLS = 1.0
LAMBDA_BOX = 10.0


## 1. Генерация синтетического датасета (1 объект на изображение)


In [ ]:
@dataclass
class Sample:
    image: np.ndarray  # HWC uint8
    class_id: int
    box: np.ndarray    # cx, cy, w, h in [0,1]


def random_color():
    return tuple(int(x) for x in np.random.randint(40, 255, size=3))


def make_sample(img_size=IMG_SIZE) -> Sample:
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    img[:] = np.random.randint(0, 40, size=3)  # тёмный фон
    cls = random.randint(0, 2)
    color = random_color()

    # размер и центр фигуры
    size = random.randint(img_size // 6, img_size // 3)
    cx = random.randint(size + 2, img_size - size - 2)
    cy = random.randint(size + 2, img_size - size - 2)

    if cls == 0:  # circle
        cv2.circle(img, (cx, cy), size, color, -1)
        x1, y1, x2, y2 = cx - size, cy - size, cx + size, cy + size
    elif cls == 1:  # rectangle
        w, h = size, int(size * random.uniform(0.6, 1.4))
        x1, y1, x2, y2 = cx - w // 2, cy - h // 2, cx + w // 2, cy + h // 2
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    else:  # triangle
        pts = np.array([
            [cx, cy - size],
            [cx - size, cy + size],
            [cx + size, cy + size],
        ], dtype=np.int32)
        cv2.fillPoly(img, [pts], color)
        x1, y1 = pts[:, 0].min(), pts[:, 1].min()
        x2, y2 = pts[:, 0].max(), pts[:, 1].max()

    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_size - 1, x2), min(img_size - 1, y2)
    bw, bh = x2 - x1, y2 - y1
    box = np.array([
        (x1 + x2) / 2 / img_size,
        (y1 + y2) / 2 / img_size,
        bw / img_size,
        bh / img_size,
    ], dtype=np.float32)
    return Sample(img, cls, box)


# демо
s = make_sample()
plt.imshow(s.image)
plt.title(f'{CLASS_NAMES[s.class_id]} box={np.round(s.box, 2)}')
plt.axis('off'); plt.show()


In [ ]:
class ShapesDataset(Dataset):
    def __init__(self, n, seed=0):
        # фиксируем seed генератора для воспроизводимости
        samples = []
        for i in range(n):
            random.seed(seed + i)
            np.random.seed(seed + i)
            samples.append(make_sample())
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        x = torch.from_numpy(s.image).float().permute(2, 0, 1) / 255.0
        y_cls = torch.tensor(s.class_id, dtype=torch.long)
        y_box = torch.from_numpy(s.box)
        return x, y_cls, y_box


# Уменьшенные выборки для ускорения учебного прогона
train_ds = ShapesDataset(1800, seed=0)
val_ds = ShapesDataset(360, seed=10_000)
test_ds = ShapesDataset(360, seed=20_000)

use_cuda = device.type == 'cuda'
loader_kwargs = {
    'num_workers': 2,
    'pin_memory': use_cuda,
}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
print(len(train_ds), len(val_ds), len(test_ds))


## 2. Модель: backbone + головы class/box


In [ ]:
class SimpleDetector(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # Более лёгкий backbone + сохраняем пространственную информацию (4x4)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.cls_head = nn.Linear(256, num_classes)
        self.box_head = nn.Linear(256, 4)

    def forward(self, x):
        f = self.backbone(x)
        f = self.head(f)
        cls_logits = self.cls_head(f)
        box = torch.sigmoid(self.box_head(f))  # cx,cy,w,h in (0,1)
        return cls_logits, box


model = SimpleDetector().to(device)
print(model)


## 3. Loss, IoU, обучение


In [ ]:
def cxcywh_to_xyxy(box):
    # box: (..., 4) cx,cy,w,h
    cx, cy, w, h = box.unbind(-1)
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    return torch.stack([x1, y1, x2, y2], dim=-1)


def box_iou(a, b, eps=1e-6):
    a = cxcywh_to_xyxy(a)
    b = cxcywh_to_xyxy(b)
    lt = torch.max(a[..., :2], b[..., :2])
    rb = torch.min(a[..., 2:], b[..., 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    area_a = (a[..., 2] - a[..., 0]).clamp(min=0) * (a[..., 3] - a[..., 1]).clamp(min=0)
    area_b = (b[..., 2] - b[..., 0]).clamp(min=0) * (b[..., 3] - b[..., 1]).clamp(min=0)
    return inter / (area_a + area_b - inter + eps)


cls_criterion = nn.CrossEntropyLoss()
box_criterion = nn.MSELoss()  # для нормализованных координат часто даёт более быстрый рост IoU
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = total_cls = total_box = 0.0
    correct = n = 0
    iou_sum = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y_cls, y_box in loader:
            x = x.to(device, non_blocking=True)
            y_cls = y_cls.to(device, non_blocking=True)
            y_box = y_box.to(device, non_blocking=True)
            logits, pred_box = model(x)
            loss_cls = cls_criterion(logits, y_cls)
            loss_box = box_criterion(pred_box, y_box)
            loss = LAMBDA_CLS * loss_cls + LAMBDA_BOX * loss_box
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            bs = x.size(0)
            total_loss += loss.item() * bs
            total_cls += loss_cls.item() * bs
            total_box += loss_box.item() * bs
            correct += (logits.argmax(1) == y_cls).sum().item()
            iou_sum += box_iou(pred_box, y_box).sum().item()
            n += bs
    return {
        'loss': total_loss / n,
        'cls': total_cls / n,
        'box': total_box / n,
        'acc': correct / n,
        'iou': iou_sum / n,
    }


history = {'train': [], 'val': []}
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    history['train'].append(tr)
    history['val'].append(va)
    print(
        f"epoch {epoch:02d}  train_loss={tr['loss']:.3f} acc={tr['acc']:.3f} iou={tr['iou']:.3f} | "
        f"val_loss={va['loss']:.3f} acc={va['acc']:.3f} iou={va['iou']:.3f}"
    )


## 4. Графики и тест


In [ ]:
def series(split, key):
    return [h[key] for h in history[split]]

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for i, key in enumerate(['loss', 'acc', 'iou']):
    ax[i].plot(series('train', key), label='train')
    ax[i].plot(series('val', key), label='val')
    ax[i].set_title(key); ax[i].legend()
plt.tight_layout(); plt.show()

test_metrics = run_epoch(test_loader, train=False)
print('TEST:', test_metrics)


## 5. Визуализация предсказаний


In [ ]:
def draw_box(img, box, color, label):
    # img: HWC float/uint8 in [0,1] or [0,255]
    out = img.copy()
    if out.dtype != np.uint8:
        out = (out * 255).astype(np.uint8)
    h, w = out.shape[:2]
    cx, cy, bw, bh = box
    x1 = int((cx - bw / 2) * w)
    y1 = int((cy - bh / 2) * h)
    x2 = int((cx + bw / 2) * w)
    y2 = int((cy + bh / 2) * h)
    cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
    cv2.putText(out, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return out


model.eval()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()
for i in range(8):
    x, y_cls, y_box = test_ds[i]
    with torch.no_grad():
        logits, pred_box = model(x.unsqueeze(0).to(device))
    pred_cls = int(logits.argmax(1).item())
    pb = pred_box.squeeze(0).cpu().numpy()
    gb = y_box.numpy()
    img = x.permute(1, 2, 0).numpy()
    vis = draw_box(img, gb, (0, 255, 0), f'GT:{CLASS_NAMES[int(y_cls)]}')
    vis = draw_box(vis, pb, (255, 0, 0), f'PR:{CLASS_NAMES[pred_cls]}')
    axes[i].imshow(vis)
    axes[i].axis('off')
plt.suptitle('GT=green, Pred=red')
plt.tight_layout(); plt.show()


**Вопрос 1.** Почему для учебной детекции удобен синтетический датасет?

**Ответ:**

---

**Вопрос 2.** Что будет, если `λ_box` слишком мал?

**Ответ:**

---

**Вопрос 3.** Чем IoU лучше простой MSE по координатам как метрика качества рамки?

**Ответ:**
